In [ ]:
# Colab 選用 2026.07；在獨立環境安裝本節套件。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用本書環境或 Colab 2026.07"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch01-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["rdkit==2026.9.1", "matplotlib==3.11.2", "numpy==1.26.4", "pillow==13.0.0"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "inchi.py"


# 三組分子對照
依序執行以下兩格。每組先看分子式，再比較 InChIKey 的首段和全碼。要查看某一對資料從哪一層開始不同，可直接比對印出的完整 InChI。

In [ ]:
%%writefile "{script}"
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from rdkit import Chem
from rdkit.Chem import inchi

def identifiers(smiles):
    mol = Chem.MolFromSmiles(smiles)
    standard = inchi.MolToInchi(mol)
    key = inchi.InchiToInchiKey(standard)
    return standard, key

pairs = [
    ("Ethanol / ethanol", "CCO", "OCC"),
    ("Ethanol / dimethyl ether", "CCO", "COC"),
    ("(S)- / (R)-lactic acid", "C[C@H](O)C(=O)O", "C[C@@H](O)C(=O)O"),
]

matches = []
for label, left, right in pairs:
    print(label)
    left_inchi, left_key = identifiers(left)
    right_inchi, right_key = identifiers(right)
    print(left, left_inchi, left_key, sep=" | ")
    print(right, right_inchi, right_key, sep=" | ")
    matches.append([
        left_inchi.split("/")[1] == right_inchi.split("/")[1],
        left_key.split("-")[0] == right_key.split("-")[0],
        left_key == right_key,
    ])

fig, ax = plt.subplots(figsize=(8.5, 3.1), layout="constrained")
ax.imshow(matches, cmap=ListedColormap(["#e4e6e9", "#236967"]), vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(3), ["Formula", "First 14 letters", "Full InChIKey"])
ax.set_yticks(range(3), [pair[0] for pair in pairs])
ax.xaxis.tick_top()
ax.tick_params(length=0, pad=8)
for row, results in enumerate(matches):
    for col, same in enumerate(results):
        ax.text(col, row, "Same" if same else "Different", ha="center", va="center",
                color="white" if same else "#292e31", fontsize=13)
for spine in ax.spines.values():
    spine.set_visible(False)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "01-03-inchi-and-inchikey.png", dpi=160)
plt.close(fig)

In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.stderr:
    print(run.stderr)
run.check_returncode()
print(run.stdout.strip())
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(out / "01-03-inchi-and-inchikey.png")))